In [1]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu peft -q
!pip uninstall -y hf_xet -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, TaskType
import time

def load_with_retry(load_fn, *args, retries=5, delay=8, **kwargs):
    for attempt in range(retries):
        try:
            return load_fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"download failed (attempt {attempt+1}/{retries}), retrying in {delay}s...")
            time.sleep(delay)
OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.8 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [2]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

=== null counts ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== answer distribution ===
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

=== prompt length (words) ===
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt, dtype: float64

=== avg option length (words) ===
  A: 26.1
  B: 26.5
  C: 26.5
  D: 26.0
  E: 26.2


In [3]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

unique vocab in cleaned prompts: 859
sample — original : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
sample — cleaned  : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


In [4]:
# from datasets import Dataset, Dataset as MCQDataset
# from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

# hf_train = Dataset.from_pandas(train)
# hf_test  = Dataset.from_pandas(test)
# print(hf_train)

# # tfidf on full corpus so test vocab is covered (no pretrained weights)
# full_corpus = []
# for df in [train, test]:
#     for _, row in df.iterrows():
#         full_corpus.append(row['prompt'])
#         for o in OPTS:
#             full_corpus.append(row[o])
# tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
# tfidf.fit(full_corpus)
# print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# # word2vec on cleaned stopwordremoved tokens from scratch
# w2v_corpus = []
# for df in [train, test]:
#     for _, row in df.iterrows():
#         w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
#         for o in OPTS:
#             w2v_corpus.append(remove_stopwords(tokenize(row[o])))
# w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
# print(f"w2v vocab: {len(w2v_model.wv)}")

# # roberta pretrained model used both for cosine sim baseline and attention inspection
# bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
# bert_model     = AutoModel.from_pretrained("roberta-base", output_attentions=True).to(device)
# bert_model.eval()
# sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
# with torch.no_grad():
#     sample_out = bert_model(**sample_inputs)
# print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
# print("this is how roberta learns which words to focus on relative to others")

# # data formatting for mcq
# # concatenating question with all 5 options into a single input string, model predicts the correct letter
# LETTER2ID = {o: i for i, o in enumerate(OPTS)}
# ID2LETTER = {i: o for o, i in LETTER2ID.items()}

# def format_mcq(row):
#     return (f"{row['prompt']} "
#             f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}")

# train_inputs = [format_mcq(row) for _, row in train.iterrows()]
# train_labels = [LETTER2ID[a] for a in train['answer']]
# print(f"\nsample formatted input: {train_inputs[0][:150]}...")

# # reusing bert_tokenizer from roberta-base,
# cls_tok  = bert_tokenizer
# cls_base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

# cls_enc = cls_tok(train_inputs, truncation=True, max_length=256, padding=False)
# cls_enc['labels'] = train_labels
# hf_cls  = MCQDataset.from_dict(cls_enc)
# print(f"training examples: {len(hf_cls)}")

# # lora freezes the base model and only trains small rank decomposition matrices added to attention q and v
# # this cuts trainable params and gpu memory drastically while reaching similar accuracy
# full_params = sum(p.numel() for p in cls_base.parameters())
# print(f"full model params (if full finetuning): {full_params:,}")

# lora_cfg = LoraConfig(
#     task_type      = TaskType.SEQ_CLS,
#     r              = 16,
#     lora_alpha     = 32,
#     target_modules = ["query", "value"],
#     lora_dropout   = 0.1,
#     bias           = "none"
# )

# lora_model = get_peft_model(cls_base, lora_cfg).to(device)
# lora_model.print_trainable_parameters()

# cls_collator = DataCollatorWithPadding(cls_tok)

# # fp16 halves memory per tensor so bigger batch can be used
# # gradient accumulationsteps simulates a larger effective batch without needing more vram
# cls_args = TrainingArguments(
#     output_dir                  = "/kaggle/working/lora_cls",
#     num_train_epochs            = 6,
#     per_device_train_batch_size = 16,
#     gradient_accumulation_steps = 2,
#     learning_rate               = 2e-4,
#     warmup_steps                = 50,
#     fp16                        = True,
#     logging_steps               = 50,
#     save_strategy               = "no",
#     report_to                   = "none",
#     dataloader_pin_memory       = False,
#     dataloader_num_workers      = 0,
#     seed                        = 42
# )

# cls_trainer = Trainer(
#     model            = lora_model,
#     args             = cls_args,
#     train_dataset    = hf_cls,
#     data_collator    = cls_collator,
#     processing_class = cls_tok
# )

# print("starting lora classification finetuning...")
# cls_trainer.train()
# lora_model.eval()
# print("lora finetuning done")

# # sanity check predictions should vary across rows, not stuck on one letter
# print("\nsanity check on first 5 train rows:")
# for i in range(5):
#     row_i  = train.iloc[i]
#     text_i = format_mcq(row_i)
#     ins_i  = cls_tok(text_i, return_tensors="pt", truncation=True, max_length=256).to(device)
#     with torch.no_grad():
#         logits_i = lora_model(**ins_i).logits[0]
#     pred_i = ID2LETTER[int(torch.argmax(logits_i))]
#     print(f"row {i} | predicted: '{pred_i}' | actual: {row_i['answer']}")

In [5]:
# def tfidf_cosine(text1, text2):
#     v1 = tfidf.transform([text1])
#     v2 = tfidf.transform([text2])
#     return float(cosine_similarity(v1, v2)[0][0])

# def avg_w2v(text):
#     tokens = remove_stopwords(tokenize(text))
#     vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
#     return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

# def w2v_cosine(text1, text2):
#     v1 = avg_w2v(text1).reshape(1, -1)
#     v2 = avg_w2v(text2).reshape(1, -1)
#     if np.all(v1 == 0) or np.all(v2 == 0):
#         return 0.0
#     return float(cosine_similarity(v1, v2)[0][0])

# # roberta mean pooling ignores padding tokens via attention mask
# def get_roberta_embedding(text):
#     inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
#     with torch.no_grad():
#         out    = bert_model(**inputs)
#     token_embs = out.last_hidden_state
#     mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
#     summed     = torch.sum(token_embs * mask, dim=1)
#     counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
#     return (summed / counts).squeeze(0).cpu().numpy()

# def rank_tfidf(row):
#     sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
#     return sorted(sims, key=lambda x: sims[x], reverse=True)

# def rank_w2v(row):
#     sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
#     return sorted(sims, key=lambda x: sims[x], reverse=True)

# def rank_roberta(row):
#     pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
#     sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
#     return sorted(sims, key=lambda x: sims[x], reverse=True)

# # lora classification ranking, logits give score per option directly
# def rank_lora(row):
#     text   = format_mcq(row)
#     inputs = cls_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
#     with torch.no_grad():
#         logits = lora_model(**inputs).logits[0]
#     scores = {ID2LETTER[i]: logits[i].item() for i in range(5)}
#     return sorted(scores, key=lambda x: scores[x], reverse=True)

# def apk(actual, predicted, k=3):
#     predicted = predicted[:k]
#     if actual not in predicted:
#         return 0.0
#     score, hits = 0.0, 0
#     for i, p in enumerate(predicted):
#         if p == actual:
#             hits  += 1
#             score += hits / (i + 1.0)
#     return score

# def mapk(actuals, predictions, k=3):
#     return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

# from sklearn.metrics import accuracy_score, f1_score

# # accuracy and f1 computed on top-1 prediction 
# # separate from mAP@3 which measures ranking quality across top-3
# def top1_accuracy(actuals, predictions):
#     top1_preds = [p[0] for p in predictions]
#     return round(accuracy_score(actuals, top1_preds), 4)

# def top1_f1(actuals, predictions):
#     top1_preds = [p[0] for p in predictions]
#     return round(f1_score(actuals, top1_preds, average='macro'), 4)

# actuals        = train['answer'].tolist()
# top3_maj       = list(train['answer'].value_counts().index[:3])
# map3_maj       = round(mapk(actuals, [top3_maj] * len(train)), 4)
# tfidf_preds    = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
# map3_tfidf     = round(mapk(actuals, tfidf_preds), 4)
# acc_tfidf = top1_accuracy(actuals, tfidf_preds)
# f1_tfidf  = top1_f1(actuals, tfidf_preds)

# w2v_preds      = [rank_w2v(row)[:3] for _, row in train.iterrows()]
# map3_w2v       = round(mapk(actuals, w2v_preds), 4)
# acc_w2v = top1_accuracy(actuals, w2v_preds)
# f1_w2v  = top1_f1(actuals, w2v_preds)

# sample_roberta = train.sample(n=200, random_state=42)
# roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
# map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)
# acc_roberta = top1_accuracy(sample_roberta['answer'].tolist(), roberta_preds)
# f1_roberta  = top1_f1(sample_roberta['answer'].tolist(), roberta_preds)

# print("evaluating lora on full train...")
# lora_preds = [rank_lora(row)[:3] for _, row in train.iterrows()]
# map3_lora  = round(mapk(actuals, lora_preds), 4)
# acc_lora = top1_accuracy(actuals, lora_preds)
# f1_lora  = top1_f1(actuals, lora_preds)

# print(f"\nmajority baseline    MAP@3 : {map3_maj}")
# print(f"tfidf cosine         MAP@3 : {map3_tfidf}")
# print(f"word2vec cosine      MAP@3 : {map3_w2v}")
# print(f"roberta cosine (200) MAP@3 : {map3_roberta}")
# print(f"lora finetuned       MAP@3 : {map3_lora}")

# print(f"\ntfidf   — acc: {acc_tfidf}, f1: {f1_tfidf}")
# print(f"w2v     — acc: {acc_w2v}, f1: {f1_w2v}")
# print(f"roberta — acc: {acc_roberta}, f1: {f1_roberta}")
# print(f"lora    — acc: {acc_lora}, f1: {f1_lora}")

# run = wandb.init(
#     project = "23f1000362-t22026",
#     name    = "milestone4-final-4models",
#     config  = {
#         "milestone": 4, "lora_r": 16, "lora_alpha": 32,
#         "epochs": 6, "lr": 2e-4, "fp16": True, "base_model": "roberta-base"
#     },
#     tags = ["milestone-4", "lora", "final"]
# )
# wandb.log({
#     "map3/majority_baseline"  : map3_maj,
#     "map3/tfidf_cosine"       : map3_tfidf,
#     "map3/word2vec_cosine"    : map3_w2v,
#     "map3/roberta_cosine_200" : map3_roberta,
#     "map3/lora_finetuned"     : map3_lora,
#     "accuracy/tfidf"          : acc_tfidf,
#     "accuracy/word2vec"       : acc_w2v,
#     "accuracy/roberta"        : acc_roberta,
#     "accuracy/lora"           : acc_lora,
#     "f1/tfidf"                : f1_tfidf,
#     "f1/word2vec"             : f1_w2v,
#     "f1/roberta"              : f1_roberta,
#     "f1/lora"                 : f1_lora
# })
# wandb.finish()


# test_preds = [" ".join(rank_lora(row)[:3]) for _, row in test.iterrows()] # lora as submission model coz highest score, 
# submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
# submission.to_csv("submission.csv", index=False)
# print("submission.csv saved")
# submission.head(5)

In [6]:
from datasets import Dataset, Dataset as MCQDataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
import torch.nn.functional as F

hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf using char n-grams
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec on cleaned stopword-removed tokens from scratch
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))
w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta pretrained model used both for cosine sim baseline and attention inspection
bert_tokenizer = load_with_retry(AutoTokenizer.from_pretrained, "roberta-base")
bert_model     = load_with_retry(AutoModel.from_pretrained, "roberta-base", output_attentions=True)
bert_model     = bert_model.to(device)
bert_model.eval()
sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")


# tfidf nn - content-based features into a trained MLP classifier
def jaccard(text1, text2):
    s1, s2 = set(str(text1).lower().split()), set(str(text2).lower().split())
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def build_features(row):
    pv = tfidf.transform([row['prompt']])
    feats = {}
    for o in OPTS:
        ov  = tfidf.transform([row[o]])
        cos = float(cosine_similarity(pv, ov)[0][0])
        jac = jaccard(row['prompt'], row[o])
        feats[o] = [cos, jac]
    return feats

X_rows, y_rows, group_ids = [], [], []
print("\nbuilding content-based features for tfidf nn classifier...")
for idx, row in train.iterrows():
    feats = build_features(row)
    for o in OPTS:
        X_rows.append(feats[o])
        y_rows.append(1 if o == row['answer'] else 0)
        group_ids.append(idx)

X_tfidf_nn = np.array(X_rows)
y_tfidf_nn = np.array(y_rows)
group_ids_arr = np.array(group_ids)
print(f"feature matrix: {X_tfidf_nn.shape}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_scores = np.zeros(len(y_tfidf_nn))
unique_groups = train.index.values

for tr_idx, val_idx in kf.split(unique_groups):
    tr_groups = set(unique_groups[tr_idx])
    va_groups = set(unique_groups[val_idx])
    tr_mask = np.array([g in tr_groups for g in group_ids_arr])
    va_mask = np.array([g in va_groups for g in group_ids_arr])
    mlp_fold = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42, early_stopping=True)
    mlp_fold.fit(X_tfidf_nn[tr_mask], y_tfidf_nn[tr_mask])
    oof_scores[va_mask] = mlp_fold.predict_proba(X_tfidf_nn[va_mask])[:, 1]

print("tfidf nn cross-validated training done")

tfidf_nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42, early_stopping=True)
tfidf_nn_model.fit(X_tfidf_nn, y_tfidf_nn)
print("final tfidf nn classifier fit on full train data")


# data formatting for mcq

LETTER2ID = {o: i for i, o in enumerate(OPTS)}
ID2LETTER  = {i: o for o, i in LETTER2ID.items()}

def format_mcq(row):
    return (f"{row['prompt']} "
            f"A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}")

train_inputs = [format_mcq(row) for _, row in train.iterrows()]
train_labels = [LETTER2ID[a] for a in train['answer']]
print(f"\nsample formatted input: {train_inputs[0][:150]}...")


# lora roberta - back to r=16, but now with a real validation split
# and early stopping so training halts before it starts memorizing
# the r=32/8-epoch run hit train loss 0.068 (basically memorized), that killed test score
cls_tok  = bert_tokenizer
cls_base = load_with_retry(AutoModelForSequenceClassification.from_pretrained, "roberta-base", num_labels=5)

cls_enc = cls_tok(train_inputs, truncation=True, max_length=256, padding=False)
cls_enc['labels'] = train_labels
hf_cls_full = MCQDataset.from_dict(cls_enc)

# holding out 10% purely to detect overfitting during training, not used for final predictions
hf_split     = hf_cls_full.train_test_split(test_size=0.1, seed=42)
hf_cls_train = hf_split['train']
hf_cls_val   = hf_split['test']
print(f"train: {len(hf_cls_train)} | val: {len(hf_cls_val)}")

full_params = sum(p.numel() for p in cls_base.parameters())
print(f"full model params (if full finetuning): {full_params:,}")

lora_cfg = LoraConfig(
    task_type      = TaskType.SEQ_CLS,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none"
)

lora_model = get_peft_model(cls_base, lora_cfg).to(device)
lora_model.print_trainable_parameters()

cls_collator = DataCollatorWithPadding(cls_tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

# early stopping watches val accuracy, stops if it doesnt improve for 2 straight evals
# this directly prevents the memorization we just saw with the r=32 8-epoch run
from transformers import EarlyStoppingCallback

cls_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_cls",
    num_train_epochs            = 12,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    gradient_accumulation_steps = 2,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 50,
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    report_to                   = "none",
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
    seed                        = 42
)

cls_trainer = Trainer(
    model            = lora_model,
    args             = cls_args,
    train_dataset    = hf_cls_train,
    eval_dataset     = hf_cls_val,
    data_collator    = cls_collator,
    processing_class = cls_tok,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("starting lora roberta finetuning (r=16, up to 12 epochs, early stopping on val accuracy)...")
cls_trainer.train()
lora_model.eval()
print("lora roberta finetuning done — using best checkpoint by validation accuracy, not the last one")

final_val = cls_trainer.evaluate()
print(f"final validation accuracy: {final_val['eval_accuracy']:.4f}")

print("\nsanity check on first 5 train rows:")
for i in range(5):
    row_i = train.iloc[i]
    ins_i = cls_tok(format_mcq(row_i), return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits_i = lora_model(**ins_i).logits[0]
    pred_i = ID2LETTER[int(torch.argmax(logits_i))]
    print(f"row {i} | predicted: '{pred_i}' | actual: {row_i['answer']}")


# milestone 5 - second model, bert-base-uncased, also with early stopping this time
# earlier bert run only trained 6 epochs, loss barely moved (6.16 -> 5.15), likely underfit
# giving it the same val split + early stopping treatment so it actually converges properly
def train_lora_on_base_with_val(base_model_name, seed=42):
    tok_v  = load_with_retry(AutoTokenizer.from_pretrained, base_model_name)
    base_v = load_with_retry(AutoModelForSequenceClassification.from_pretrained, base_model_name, num_labels=5)

    enc_v = tok_v(train_inputs, truncation=True, max_length=256, padding=False)
    enc_v['labels'] = train_labels
    hf_v_full = MCQDataset.from_dict(enc_v)
    hf_v_split = hf_v_full.train_test_split(test_size=0.1, seed=seed)

    lora_cfg_v = LoraConfig(
        task_type      = TaskType.SEQ_CLS,
        r              = 16,
        lora_alpha     = 32,
        target_modules = ["query", "value"],
        lora_dropout   = 0.1,
        bias           = "none"
    )
    lora_model_v = get_peft_model(base_v, lora_cfg_v).to(device)
    collator_v   = DataCollatorWithPadding(tok_v)

    args_v = TrainingArguments(
        output_dir                  = f"/kaggle/working/lora_{base_model_name.replace('/', '_')}",
        num_train_epochs            = 12,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        gradient_accumulation_steps = 2,
        learning_rate               = 2e-4,
        weight_decay                = 0.01,
        warmup_steps                = 50,
        fp16                        = True,
        logging_steps                = 100,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "accuracy",
        greater_is_better           = True,
        report_to                   = "none",
        dataloader_pin_memory       = False,
        dataloader_num_workers      = 0,
        seed                        = seed
    )
    trainer_v = Trainer(
        model            = lora_model_v,
        args             = args_v,
        train_dataset    = hf_v_split['train'],
        eval_dataset     = hf_v_split['test'],
        data_collator    = collator_v,
        processing_class = tok_v,
        compute_metrics  = compute_metrics,
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
    )
    trainer_v.train()
    lora_model_v.eval()
    val_metrics = trainer_v.evaluate()
    return lora_model_v, tok_v, val_metrics['eval_accuracy']

print("\ntraining lora on bert-base-uncased with early stopping...")
lora_model_bert, tok_bert, bert_val_acc = train_lora_on_base_with_val("bert-base-uncased")
print(f"bert-base-uncased done — val accuracy: {bert_val_acc:.4f}")

def quick_sample_accuracy(model, tok, sample_df):
    correct = 0
    for _, row in sample_df.iterrows():
        ins = tok(format_mcq(row), return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            pred = ID2LETTER[int(torch.argmax(model(**ins).logits[0]))]
        if pred == row['answer']:
            correct += 1
    return correct / len(sample_df)

ens_sample  = train.sample(n=300, random_state=99)
acc_roberta = quick_sample_accuracy(lora_model, cls_tok, ens_sample)
acc_bert    = quick_sample_accuracy(lora_model_bert, tok_bert, ens_sample)
print(f"\nsample accuracy — roberta: {acc_roberta:.4f}, bert: {acc_bert:.4f}")

ensemble_members = [
    (lora_model, cls_tok, acc_roberta),
    (lora_model_bert, tok_bert, acc_bert)
]
print(f"ensemble ready: {len(ensemble_members)} models")

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'clean_prompt', 'clean_A', 'clean_B', 'clean_C', 'clean_D', 'clean_E'],
    num_rows: 2000
})
tfidf vocab: 23905
w2v vocab: 2883


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'eval_loss': '3.166', 'eval_accuracy': '0.245', 'eval_runtime': '1.977', 'eval_samples_per_second': '101.2', 'eval_steps_per_second': '2.023', 'epoch': '1'}
{'eval_loss': '3.009', 'eval_accuracy': '0.34', 'eval_runtime': '1.942', 'eval_samples_per_second': '103', 'eval_steps_per_second': '2.06', 'epoch': '2'}
{'eval_loss': '2.375', 'eval_accuracy': '0.565', 'eval_runtime': '1.982', 'eval_samples_per_second': '100.9', 'eval_steps_per_second': '2.018', 'epoch': '3'}
{'loss': '5.872', 'grad_norm': '12.05', 'learning_rate': '0.0001671', 'epoch': '3.456'}
{'eval_loss': '1.95', 'eval_accuracy': '0.515', 'eval_runtime': '1.944', 'eval_samples_per_second': '102.9', 'eval_steps_per_second': '2.058', 'epoch': '4'}
{'eval_loss': '1.391', 'eval_accuracy': '0.785', 'eval_runtime': '1.97', 'eval_samples_per_second': '101.5', 'eval_steps_per_second': '2.03', 'epoch': '5'}
{'eval_loss': '1.198', 'eval_accuracy': '0.795', 'eval_runtime': '1.966', 'eval_samples_per_second': '101.8', 'eval_steps_per_sec

In [7]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

def get_roberta_embedding(text):
    inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out    = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed     = torch.sum(token_embs * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_tfidf_nn(row):
    feats = build_features(row)
    scores = {o: tfidf_nn_model.predict_proba([feats[o]])[0][1] for o in OPTS}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_lora(row):
    text   = format_mcq(row)
    inputs = cls_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = lora_model(**inputs).logits[0]
    scores = {ID2LETTER[i]: logits[i].item() for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def get_member_probs(model, tok, text):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    return F.softmax(logits, dim=-1).cpu().numpy()

def rank_ensemble_weighted(row):
    text        = format_mcq(row)
    probs_rob   = get_member_probs(lora_model, cls_tok, text)
    probs_bert  = get_member_probs(lora_model_bert, tok_bert, text)
    total_acc   = acc_roberta + acc_bert
    w_rob       = acc_roberta / total_acc
    w_bert      = acc_bert / total_acc
    final_probs = w_rob * probs_rob + w_bert * probs_bert
    scores      = {ID2LETTER[i]: final_probs[i] for i in range(5)}
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def rank_fusion(row, w_rob=0.6, w_bert=0.4):
    text       = format_mcq(row)
    probs_rob  = get_member_probs(lora_model, cls_tok, text)
    probs_bert = get_member_probs(lora_model_bert, tok_bert, text)
    rank_rob   = {ID2LETTER[i]: r for r, i in enumerate(np.argsort(-probs_rob))}
    rank_bert  = {ID2LETTER[i]: r for r, i in enumerate(np.argsort(-probs_bert))}
    fused = {o: w_rob * rank_rob[o] + w_bert * rank_bert[o] for o in OPTS}
    return sorted(fused, key=lambda x: fused[x])

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

from sklearn.metrics import accuracy_score, f1_score

def top1_accuracy(actuals, predictions):
    return round(accuracy_score(actuals, [p[0] for p in predictions]), 4)

def top1_f1(actuals, predictions):
    return round(f1_score(actuals, [p[0] for p in predictions], average='macro'), 4)

actuals     = train['answer'].tolist()
top3_maj    = list(train['answer'].value_counts().index[:3])
map3_maj    = round(mapk(actuals, [top3_maj] * len(train)), 4)

tfidf_preds = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf  = round(mapk(actuals, tfidf_preds), 4)
acc_tfidf   = top1_accuracy(actuals, tfidf_preds)
f1_tfidf    = top1_f1(actuals, tfidf_preds)

tfidf_nn_preds = []
oof_idx = 0
for _, row in train.iterrows():
    scores = {o: oof_scores[oof_idx + i] for i, o in enumerate(OPTS)}
    oof_idx += 5
    tfidf_nn_preds.append(sorted(scores, key=lambda x: scores[x], reverse=True)[:3])
map3_tfidf_nn = round(mapk(actuals, tfidf_nn_preds), 4)
acc_tfidf_nn  = top1_accuracy(actuals, tfidf_nn_preds)
f1_tfidf_nn   = top1_f1(actuals, tfidf_nn_preds)

w2v_preds   = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v    = round(mapk(actuals, w2v_preds), 4)
acc_w2v     = top1_accuracy(actuals, w2v_preds)
f1_w2v      = top1_f1(actuals, w2v_preds)

sample_roberta  = train.sample(n=200, random_state=42)
roberta_preds   = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta    = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)
acc_roberta_cos = top1_accuracy(sample_roberta['answer'].tolist(), roberta_preds)
f1_roberta_cos  = top1_f1(sample_roberta['answer'].tolist(), roberta_preds)

# evaluating lora ONLY on the held-out val set now, not full train
# this is the honest number since the model never saw these rows during training
val_rows = train.iloc[hf_cls_val.__len__() * -1:] if False else None  # placeholder, using proper val indices below

print("evaluating lora roberta on held-out validation split (honest, not memorized)...")
val_indices = hf_split['test']['labels']  # just to confirm split exists
# rebuilding val row lookup from the split done in cell 4
val_df = train.iloc[[i for i in range(len(train))]].copy()
# since datasets shuffles internally, easiest honest check is the trainer's own eval_accuracy
print(f"roberta validation accuracy (from trainer): {final_val['eval_accuracy']:.4f}")
print(f"bert validation accuracy (from trainer): {bert_val_acc:.4f}")

print("\nevaluating lora roberta on full train (reference only, will look inflated)...")
lora_preds = [rank_lora(row)[:3] for _, row in train.iterrows()]
map3_lora  = round(mapk(actuals, lora_preds), 4)
acc_lora   = top1_accuracy(actuals, lora_preds)
f1_lora    = top1_f1(actuals, lora_preds)

print("evaluating weighted ensemble on full train (reference only)...")
ens_preds = [rank_ensemble_weighted(row)[:3] for _, row in train.iterrows()]
map3_ens  = round(mapk(actuals, ens_preds), 4)
acc_ens   = top1_accuracy(actuals, ens_preds)
f1_ens    = top1_f1(actuals, ens_preds)

print("evaluating rank fusion on full train (reference only)...")
rankfusion_preds = [rank_fusion(row)[:3] for _, row in train.iterrows()]
map3_rankfusion   = round(mapk(actuals, rankfusion_preds), 4)
acc_rankfusion    = top1_accuracy(actuals, rankfusion_preds)
f1_rankfusion     = top1_f1(actuals, rankfusion_preds)

print(f"\nmajority baseline         MAP@3 : {map3_maj}")
print(f"tfidf cosine              MAP@3 : {map3_tfidf}")
print(f"tfidf + nn (OOF, honest)  MAP@3 : {map3_tfidf_nn}")
print(f"word2vec cosine           MAP@3 : {map3_w2v}")
print(f"roberta cosine (200)      MAP@3 : {map3_roberta}")
print(f"roberta VAL accuracy (honest)  : {final_val['eval_accuracy']:.4f}")
print(f"bert VAL accuracy (honest)     : {bert_val_acc:.4f}")
print(f"lora roberta (full train, inflated) MAP@3 : {map3_lora}")
print(f"weighted ensemble (full train, inflated) MAP@3 : {map3_ens}")
print(f"rank fusion (full train, inflated) MAP@3 : {map3_rankfusion}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone5-earlystop-ensemble",
    config  = {
        "milestone": 5, "lora_r": 16, "lora_alpha": 32,
        "max_epochs": 12, "early_stopping_patience": 2,
        "ensemble": ["roberta-base", "bert-base-uncased"],
        "note": "fixed overfitting from r=32/8epoch run via val split + early stopping"
    },
    tags = ["milestone-5", "lora", "ensemble", "early-stopping"]
)
wandb.log({
    "map3/majority_baseline"     : map3_maj,
    "map3/tfidf_cosine"          : map3_tfidf,
    "map3/tfidf_nn_oof"          : map3_tfidf_nn,
    "map3/word2vec_cosine"       : map3_w2v,
    "map3/roberta_cosine_200"    : map3_roberta,
    "accuracy/roberta_val_honest": final_val['eval_accuracy'],
    "accuracy/bert_val_honest"   : bert_val_acc,
    "map3/lora_fulltrain_ref"    : map3_lora,
    "map3/ensemble_fulltrain_ref": map3_ens,
    "map3/rankfusion_fulltrain_ref": map3_rankfusion
})
wandb.finish()

best_technique = "rank_fusion" if map3_rankfusion > map3_ens else "weighted"
print(f"\nusing {best_technique} ensemble for submission")

test_preds = []
for _, row in test.iterrows():
    ranked = rank_fusion(row) if best_technique == "rank_fusion" else rank_ensemble_weighted(row)
    test_preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head(5)

evaluating lora roberta on held-out validation split (honest, not memorized)...
roberta validation accuracy (from trainer): 1.0000
bert validation accuracy (from trainer): 0.8800

evaluating lora roberta on full train (reference only, will look inflated)...
evaluating weighted ensemble on full train (reference only)...
evaluating rank fusion on full train (reference only)...

majority baseline         MAP@3 : 0.4213
tfidf cosine              MAP@3 : 0.3105
tfidf + nn (OOF, honest)  MAP@3 : 0.4176
word2vec cosine           MAP@3 : 0.3147
roberta cosine (200)      MAP@3 : 0.3425
roberta VAL accuracy (honest)  : 1.0000
bert VAL accuracy (honest)     : 0.8800
lora roberta (full train, inflated) MAP@3 : 1.0
weighted ensemble (full train, inflated) MAP@3 : 1.0
rank fusion (full train, inflated) MAP@3 : 1.0

using weighted ensemble for submission
submission.csv saved


,ID,Prediction
0,1,A C E
1,2,B C E
2,3,B C E
3,4,E A D
4,5,C B A
